In [14]:
import glob
import json
import pandas as pd

raw_data_path = '../data/raw'
clean_data_path = '../data/clean'

# Tratamento de Despesas

In [15]:
despesas_df = pd.DataFrame()

for f in glob.glob(f'{raw_data_path}/despesas-pela-cota-para-exercício-da-atividade-parlamentar/*.csv.zip'):
    print(f)
    despesas_df = pd.concat([despesas_df, pd.read_csv(f, sep=';')], ignore_index=True)

despesas_df.reset_index(drop=True)

despesas_df = despesas_df[[
    'txNomeParlamentar',
    'sgPartido',
    'txtDescricao',
    'txtDescricaoEspecificacao',
    'datEmissao',
    'vlrLiquido',
    'numMes',
    'numAno',
    'ideCadastro',
]]

despesas_df = despesas_df.rename(columns={
    'txNomeParlamentar': 'nome_parlamentar',
    'sgPartido': 'sigla_partido',
    'txtDescricao': 'descricao',
    'txtDescricaoEspecificacao': 'descricao_especificacao',
    'datEmissao': 'data_emissao',
    'vlrLiquido': 'valor_liquido',
    'numMes': 'mes',
    'numAno': 'ano',
    'ideCadastro': 'deputado_id',
})


despesas_df.to_parquet(
    f'{clean_data_path}/despesa-para-exercicio-da-atividade-parlamentar/despesa-para-exercicio-da-atividade-parlamentar.parquet',
)

despesas_df.sort_values('valor_liquido', ascending=False).head()

../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2023.csv.zip


/tmp/ipykernel_14816/4281838870.py:5: DtypeWarning: Columns (27: datPagamentoRestituicao) have mixed types. Specify dtype option on import or set low_memory=False.
  despesas_df = pd.concat([despesas_df, pd.read_csv(f, sep=';')], ignore_index=True)


../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2026.csv.zip


/tmp/ipykernel_14816/4281838870.py:5: DtypeWarning: Columns (23: txtPassageiro, 24: txtTrecho, 27: datPagamentoRestituicao) have mixed types. Specify dtype option on import or set low_memory=False.
  despesas_df = pd.concat([despesas_df, pd.read_csv(f, sep=';')], ignore_index=True)


../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2025.csv.zip


/tmp/ipykernel_14816/4281838870.py:5: DtypeWarning: Columns (27: datPagamentoRestituicao) have mixed types. Specify dtype option on import or set low_memory=False.
  despesas_df = pd.concat([despesas_df, pd.read_csv(f, sep=';')], ignore_index=True)


../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2024.csv.zip


/tmp/ipykernel_14816/4281838870.py:5: DtypeWarning: Columns (27: datPagamentoRestituicao) have mixed types. Specify dtype option on import or set low_memory=False.
  despesas_df = pd.concat([despesas_df, pd.read_csv(f, sep=';')], ignore_index=True)


,nome_parlamentar,sigla_partido,descricao,descricao_especificacao,data_emissao,valor_liquido,mes,ano,deputado_id
712849,Duda Ramos,PODE,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,NaN,2024-12-30T00:00:00,184428.0,12,2024,220540.0
114853,Gustinho Ribeiro,PP,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,NaN,2023-12-29T00:00:00,164900.0,12,2023,204408.0
159663,Alexandre Guimarães,MDB,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,NaN,2023-12-14T00:00:00,163791.0,12,2023,220542.0
665017,Professor Alcides,PSDB,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,NaN,2024-12-27T00:00:00,150960.0,12,2024,204390.0
241196,Silas Câmara,REPUBLICANOS,LOCAÇÃO OU FRETAMENTO DE AERONAVES,NaN,2026-02-19T00:00:00,149500.0,2,2026,74356.0


# Tratamento de partidos

In [16]:
with open(f'{raw_data_path}/partidos/partidos.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

    partidos_df = pd.json_normalize(data['dados']).drop(columns=['uri'])
    partidos_df['sigla'] = partidos_df['sigla'].astype(str).str.upper()
    partidos_df['nome'] = partidos_df['nome'].astype(str).str.upper()

    partidos_df = partidos_df.rename(columns={
        "id": "partido_id",
    })

partidos_df.head(n=10)

,partido_id,sigla,nome
0,36898,AVANTE,AVANTE
1,37905,CIDADANIA,CIDADANIA
2,37902,DC,DEMOCRACIA CRISTÃ
3,36899,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO
4,38011,MISSÃO,PARTIDO MISSÃO
5,37901,NOVO,PARTIDO NOVO
6,36779,PCDOB,PARTIDO COMUNISTA DO BRASIL
7,36786,PDT,PARTIDO DEMOCRÁTICO TRABALHISTA
8,37906,PL,PARTIDO LIBERAL
9,36896,PODE,PODEMOS


In [17]:
classificacao_ideologica_df = pd.read_csv(f'{raw_data_path}/classificacao-ideologica-por-partido-politico-brasileiro/classificacao_ideologica_por_partido_politico_brasileiro.csv', sep=';')

classificacao_ideologica_df['sigla'] = classificacao_ideologica_df['Partido'].astype(str).str.upper()
classificacao_ideologica_df['media'] = pd.to_numeric(classificacao_ideologica_df['Média'].astype(str).str.replace(',', '.'), errors='coerce')

macro_bins = [-float('inf'), 1.5, 3.0, 4.5, 5.5, 7.0, 8.5, float('inf')]
labels_7 = [
    'EXTREMA-ESQUERDA', 
    'ESQUERDA', 
    'CENTRO-ESQUERDA', 
    'CENTRO', 
    'CENTRO-DIREITA', 
    'DIREITA', 
    'EXTREMA-DIREITA'
]

classificacao_ideologica_df['espectro_detalhado'] = pd.cut(classificacao_ideologica_df['media'], bins=macro_bins, labels=labels_7)

# 3. Agrupamento Macro em 3 categorias (Esquerda, Centro, Direita)
bins_3 = [-float('inf'), 4.5, 5.5, float('inf')]
labels_3 = ['ESQUERDA', 'CENTRO', 'DIREITA']

classificacao_ideologica_df['espectro_macro'] = pd.cut(classificacao_ideologica_df['media'], bins=bins_3, labels=labels_3)

classificacao_ideologica_df = classificacao_ideologica_df[[
    'sigla',
    'espectro_detalhado',
    'espectro_macro'
]]

In [18]:
import pandas as pd

partido_completo_df = partidos_df.merge(
    classificacao_ideologica_df,
    left_on='sigla',
    right_on='sigla',
    how='left',
)

partido_completo_df['espectro_macro'] = (
    partido_completo_df['espectro_macro'].astype(str).replace('nan', pd.NA)
)
partido_completo_df['espectro_detalhado'] = (
    partido_completo_df['espectro_detalhado'].astype(str).replace('nan', pd.NA)
)

espectro_macro_faltantes = {
    'PL': 'DIREITA',
    'PP': 'DIREITA',  
    'CIDADANIA': 'CENTRO',  
    'PODE': 'CENTRO',  
    'PRD': 'CENTRO',  
    'MISSÃO': 'DIREITA',  
    "REPUBLICANOS": 'DIREITA',
    "SOLIDARIEDADE": 'CENTRO',
    "UNIÃO": 'DIREITA'
}

espectro_detalhado_faltantes = {
    'PL': 'DIREITA',
    'PP': 'CENTRO-DIREITA',
    'CIDADANIA': 'CENTRO-ESQUERDA',  
    'PODE': 'CENTRO',
    'PRD': 'CENTRO-DIREITA',
    'MISSÃO': 'DIREITA',  
    "REPUBLICANOS": 'CENTRO-DIREITA',
    "SOLIDARIEDADE": 'CENTRO',
    "UNIÃO": 'CENTRO-DIREITA'
}

partido_completo_df['espectro_macro'] = partido_completo_df['espectro_macro'].fillna(
    partido_completo_df['sigla'].map(espectro_macro_faltantes)
)

partido_completo_df['espectro_detalhado'] = partido_completo_df['espectro_detalhado'].fillna(
    partido_completo_df['sigla'].map(espectro_detalhado_faltantes)
)

partido_completo_df.to_parquet(f'{clean_data_path}/partido/partido.parquet')
partido_completo_df.head(n=100)

,partido_id,sigla,nome,espectro_detalhado,espectro_macro
0,36898,AVANTE,AVANTE,CENTRO-DIREITA,DIREITA
1,37905,CIDADANIA,CIDADANIA,CENTRO-ESQUERDA,CENTRO
2,37902,DC,DEMOCRACIA CRISTÃ,DIREITA,DIREITA
3,36899,MDB,MOVIMENTO DEMOCRÁTICO BRASILEIRO,DIREITA,DIREITA
4,38011,MISSÃO,PARTIDO MISSÃO,DIREITA,DIREITA
5,37901,NOVO,PARTIDO NOVO,DIREITA,DIREITA
6,36779,PCDOB,PARTIDO COMUNISTA DO BRASIL,ESQUERDA,ESQUERDA
7,36786,PDT,PARTIDO DEMOCRÁTICO TRABALHISTA,CENTRO-ESQUERDA,ESQUERDA
8,37906,PL,PARTIDO LIBERAL,DIREITA,DIREITA
9,36896,PODE,PODEMOS,CENTRO,CENTRO


# Tratamento de Proposições

In [19]:
proposicoes_df = pd.DataFrame()
for f in glob.glob(f'{raw_data_path}/proposicoes/*.csv'):
    print(f)
    proposicoes_df = pd.concat([proposicoes_df, pd.read_csv(f, sep=';')], ignore_index=True)

proposicoes_df = proposicoes_df[['id', 'descricaoTipo', 'ano']]


proposicoes_autores_df = pd.DataFrame()
for f in glob.glob(f'{raw_data_path}/proposicoes-autores/*.csv'):
    print(f)
    proposicoes_autores_df = pd.concat([proposicoes_autores_df, pd.read_csv(f, sep=';')], ignore_index=True)

proposicao_primeiro_autor = proposicoes_autores_df.loc[proposicoes_autores_df['ordemAssinatura'] == 1]

proposicao_primeiro_autor = proposicao_primeiro_autor[[
    'idProposicao',
    'idDeputadoAutor',
]]

proposicao_completa_df = pd.merge(
    proposicoes_df,
    proposicao_primeiro_autor,
    left_on='id',
    right_on='idProposicao',
    how='left'
)

proposicao_completa_df = proposicao_completa_df.drop('idProposicao', axis=1)
proposicao_completa_df = proposicao_completa_df.dropna(subset=['idDeputadoAutor'])

proposicao_completa_df.rename(columns={
    'id': 'proposicao_id',
    'descricaoTipo': 'tipo',
    'idDeputadoAutor': 'autor_deputado_id',
}, inplace=True)

proposicao_completa_df['tipo'] = proposicao_completa_df['tipo'].astype(str).str.upper()
projeto_lei_df = proposicao_completa_df.loc[proposicao_completa_df['tipo'] == 'PROJETO DE LEI']

projeto_lei_df.to_parquet(f'{clean_data_path}/projeto-lei/projeto_lei.parquet')
projeto_lei_df.head()

../data/raw/proposicoes/proposicoes-2026.csv
../data/raw/proposicoes/proposicoes-2023.csv


/tmp/ipykernel_14816/4186882622.py:4: DtypeWarning: Columns (14: uriPropPosterior) have mixed types. Specify dtype option on import or set low_memory=False.
  proposicoes_df = pd.concat([proposicoes_df, pd.read_csv(f, sep=';')], ignore_index=True)


../data/raw/proposicoes/proposicoes-2024.csv


/tmp/ipykernel_14816/4186882622.py:4: DtypeWarning: Columns (14: uriPropPosterior) have mixed types. Specify dtype option on import or set low_memory=False.
  proposicoes_df = pd.concat([proposicoes_df, pd.read_csv(f, sep=';')], ignore_index=True)


../data/raw/proposicoes/proposicoes-2025.csv
../data/raw/proposicoes-autores/proposicoesAutores-2023.csv
../data/raw/proposicoes-autores/proposicoesAutores-2024.csv
../data/raw/proposicoes-autores/proposicoesAutores-2026.csv
../data/raw/proposicoes-autores/proposicoesAutores-2025.csv


,proposicao_id,tipo,ano,autor_deputado_id
1,424197,PROJETO DE LEI,2026,73441.0
2,2168236,PROJETO DE LEI,2026,74856.0
50,2599937,PROJETO DE LEI,2026,80815.0
51,2599938,PROJETO DE LEI,2026,220701.0
52,2599939,PROJETO DE LEI,2026,220701.0


# Tratamento de Deputados

In [20]:
deputados_df = pd.read_csv(f'{raw_data_path}/deputados/br_camara_dados_abertos_deputado.csv')
deputados_df = deputados_df.drop(['data_falecimento', 'data_nascimento', 'id_municipio_nascimento', 'sigla_uf_nascimento', 'id_inicial_legislatura', 'id_final_legislatura', 'url_site', 'url_rede_social'], axis=1)

deputados_df = deputados_df[['nome', 'nome_civil', 'id_deputado', 'sexo']].rename(columns={
    'id_deputado': 'deputado_id',
})

deputados_df.to_parquet(f'{clean_data_path}/deputado/deputado.parquet')
deputados_df.head()

,nome,nome_civil,deputado_id,sexo
0,MONTEIRO DE BARROS,Antonio Augusto Monteiro de Barros,24,Masculino
1,MAIA,José Antônio da Silva Maia,23,Masculino
2,CUSTÓDIO JOSÉ DIAS,Custódio José Dias,231,Masculino
3,SOARES DA ROCHA,José Ribeiro Soares da Rocha,122,Masculino
4,LINO COUTINHO,José Lino Coutinho,117,Masculino
